In [23]:
from __future__ import annotations

import logging
import operator
import os
from typing import List, TypedDict, Annotated, Literal, Optional

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langchain_mistralai import ChatMistralAI

from langgraph.types import Send

from langchain_core.messages import SystemMessage, HumanMessage

from langchain_tavily import TavilySearch

# try:
#     from langchain_tavily import TavilySearch
# except ImportError:
#     from langchain_community.tools.tavily_search import TavilySearchResults as TavilySearch

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s [%(name)s] %(message)s",
)
logger = logging.getLogger("ai_research_copilot")
TAVILY_API_KEY_ENV = "TAVILY_API_KEY"


class ConfigurationError(RuntimeError):
    pass

In [ ]:
class Task(BaseModel):
    id: int
    title: str

    goal: str = Field(
        ...,
        description="One sentence describing what the reader should be able to do/understand after this section.",
    )
    bullets: List[str] = Field(
        ...,
        min_length=3,
        max_length=5,
        description="3–5 concrete, non-overlapping subpoints to cover in this section.",
    )
    target_words: int = Field(
        ...,
        description="Target word count for this section (120–450).",
    )
    tags: List[str] = Field(default_factory=list)
    requires_research: bool = False
    requires_citations: bool = False
    requires_code: bool = False

In [ ]:
class Plan(BaseModel):
    blog_title: str
    audience: str = Field(..., description="Who this blog is for.")
    tone: str = Field(..., description="Writing tone (e.g., practical, crisp).")
    blog_kind: Literal["explainer", "tutorial", "news_roundup", "comparison", "system_design"] = "explainer"
    constraints: List[str] = Field(default_factory=list)
    tasks: List[Task]

In [ ]:
class EvidenceItem(BaseModel):
    title: str
    url: str
    published_at: Optional[str] = None  # keep if Tavily provides; DO NOT rely on it
    snippet: Optional[str] = None
    source: Optional[str] = None

In [ ]:
class RouterDecision(BaseModel):
    needs_research: bool
    mode: Literal["closed_book", "hybrid", "open_book"]
    queries: List[str] = Field(default_factory=list)

In [ ]:
class EvidencePack(BaseModel):
    evidence: List[EvidenceItem] = Field(default_factory=list)

In [ ]:
class State(TypedDict):
    topic: str

    # routing / research
    mode: str
    needs_research: bool
    queries: List[str]
    evidence: List[EvidenceItem]
    plan: Optional[Plan]

    # workers
    sections: Annotated[List[tuple[int, str]], operator.add]  # (task_id, section_md)
    final: str

In [ ]:
llm = ChatMistralAI()

In [ ]:
ROUTER_SYSTEM = """You are a routing module for a technical blog planner.

Decide whether web research is needed BEFORE planning.

Modes:
- closed_book (needs_research=false):
  Evergreen topics where correctness does not depend on recent facts (concepts, fundamentals).
- hybrid (needs_research=true):
  Mostly evergreen but needs up-to-date examples/tools/models to be useful.
- open_book (needs_research=true):
  Mostly volatile: weekly roundups, "this week", "latest", rankings, pricing, policy/regulation.

If needs_research=true:
- Output 3–10 high-signal queries.
- Queries should be scoped and specific (avoid generic queries like just "AI" or "LLM").
- If user asked for "last week/this week/latest", reflect that constraint IN THE QUERIES.
"""

def router_node(state: State) -> dict:
    try:
        topic = state["topic"]
        logger.info("Routing topic: %s", topic)

        decider = llm.with_structured_output(RouterDecision)
        decision = decider.invoke(
            [
                SystemMessage(content=ROUTER_SYSTEM),
                HumanMessage(content=f"Topic: {topic}"),
            ]
        )

        logger.info(
            "Router decision: needs_research=%s mode=%s queries=%d",
            decision.needs_research,
            decision.mode,
            len(decision.queries),
        )
        return {
            "needs_research": decision.needs_research,
            "mode": decision.mode,
            "queries": decision.queries,
        }
    except Exception:
        logger.exception("Router node failed")
        raise

def route_next(state: State) -> str:
    return "research" if state["needs_research"] else "orchestrator"

In [ ]:
def _tavily_search(query: str, max_results: int = 5) -> List[dict]:
    try:
        logger.info("Running Tavily search: query=%s max_results=%d", query, max_results)
        tool = TavilySearch(max_results=max_results)
        response = tool.invoke({"query": query})
        results = response.get("results", response) if isinstance(response, dict) else response

        normalized: List[dict] = []
        for r in results or []:
            normalized.append(
                {
                    "title": r.get("title") or "",
                    "url": r.get("url") or "",
                    "snippet": r.get("content") or r.get("snippet") or "",
                    "published_at": r.get("published_date") or r.get("published_at"),
                    "source": r.get("source"),
                }
            )

        logger.info("Tavily search returned %d results for query=%s", len(normalized), query)
        return normalized
    except Exception:
        logger.exception("Tavily search failed for query=%s", query)
        return []


RESEARCH_SYSTEM = """You are a research synthesizer for technical writing.

Given raw web search results, produce a deduplicated list of EvidenceItem objects.

Rules:
- Only include items with a non-empty url.
- Prefer relevant + authoritative sources (company blogs, docs, reputable outlets).
- If a published date is explicitly present in the result payload, keep it as YYYY-MM-DD.
  If missing or unclear, set published_at=null. Do NOT guess.
- Keep snippets short.
- Deduplicate by URL.
"""

def research_node(state: State) -> dict:
    try:
        queries = (state.get("queries", []) or [])[:10]
        max_results = 6
        logger.info("Research node started with %d queries", len(queries))

        if queries and not os.getenv(TAVILY_API_KEY_ENV):
            message = (
                f"{TAVILY_API_KEY_ENV} is required for research mode. "
                "Set it before running the graph, for example: "
                f"os.environ[{TAVILY_API_KEY_ENV!r}] = 'your_tavily_key'"
            )
            logger.error(message)
            raise ConfigurationError(message)

        raw_results: List[dict] = []
        for q in queries:
            raw_results.extend(_tavily_search(q, max_results=max_results))

        if not raw_results:
            logger.warning("Research node found no raw results")
            return {"evidence": []}

        extractor = llm.with_structured_output(EvidencePack)
        pack = extractor.invoke(
            [
                SystemMessage(content=RESEARCH_SYSTEM),
                HumanMessage(content=f"Raw results:\n{raw_results}"),
            ]
        )

        dedup = {}
        for e in pack.evidence:
            if e.url:
                dedup[e.url] = e

        logger.info("Research node produced %d evidence items", len(dedup))
        return {"evidence": list(dedup.values())}
    except ConfigurationError as exc:
        logger.error("Research node blocked: %s", exc)
        raise
    except Exception:
        logger.exception("Research node failed")
        raise


In [ ]:
ORCH_SYSTEM = """You are a senior technical writer and developer advocate.
Your job is to produce a highly actionable outline for a technical blog post.

Hard requirements:
- Create 5–9 sections (tasks) suitable for the topic and audience.
- Each task must include:
  1) goal (1 sentence)
  2) 3–6 bullets that are concrete, specific, and non-overlapping
  3) target word count (120–550)

Quality bar:
- Assume the reader is a developer; use correct terminology.
- Bullets must be actionable: build/compare/measure/verify/debug.
- Ensure the overall plan includes at least 2 of these somewhere:
  * minimal code sketch / MWE (set requires_code=True for that section)
  * edge cases / failure modes
  * performance/cost considerations
  * security/privacy considerations (if relevant)
  * debugging/observability tips

Grounding rules:
- Mode closed_book: keep it evergreen; do not depend on evidence.
- Mode hybrid:
  - Use evidence for up-to-date examples (models/tools/releases) in bullets.
  - Mark sections using fresh info as requires_research=True and requires_citations=True.
- Mode open_book:
  - Set blog_kind = "news_roundup".
  - Every section is about summarizing events + implications.
  - DO NOT include tutorial/how-to sections unless user explicitly asked for that.
  - If evidence is empty or insufficient, create a plan that transparently says "insufficient sources"
    and includes only what can be supported.

Output must strictly match the Plan schema.
"""

def orchestrator_node(state: State) -> dict:
    try:
        planner = llm.with_structured_output(Plan)

        evidence = state.get("evidence", [])
        mode = state.get("mode", "closed_book")
        logger.info(
            "Orchestrator started: topic=%s mode=%s evidence_items=%d",
            state["topic"],
            mode,
            len(evidence),
        )

        plan = planner.invoke(
            [
                SystemMessage(content=ORCH_SYSTEM),
                HumanMessage(
                    content=(
                        f"Topic: {state['topic']}\n"
                        f"Mode: {mode}\n\n"
                        f"Evidence (ONLY use for fresh claims; may be empty):\n"
                        f"{[e.model_dump() for e in evidence][:16]}"
                    )
                ),
            ]
        )

        logger.info("Plan created: title=%s tasks=%d", plan.blog_title, len(plan.tasks))
        return {"plan": plan}
    except Exception:
        logger.exception("Orchestrator node failed")
        raise

In [ ]:
def fanout(state: State):
    try:
        logger.info("Fanout dispatching %d worker tasks", len(state["plan"].tasks))
        return [
            Send(
                "worker",
                {
                    "task": task.model_dump(),
                    "topic": state["topic"],
                    "mode": state["mode"],
                    "plan": state["plan"].model_dump(),
                    "evidence": [e.model_dump() for e in state.get("evidence", [])],
                },
            )
            for task in state["plan"].tasks
        ]
    except Exception:
        logger.exception("Fanout failed")
        raise

In [ ]:
WORKER_SYSTEM = """You are a senior technical writer and developer advocate.
Write ONE section of a technical blog post in Markdown.

Hard constraints:
- Follow the provided Goal and cover ALL Bullets in order (do not skip or merge bullets).
- Stay close to Target words (±15%).
- Output ONLY the section content in Markdown (no blog title H1, no extra commentary).
- Start with a '## <Section Title>' heading.

Scope guard:
- If blog_kind == "news_roundup": do NOT turn this into a tutorial/how-to guide.
  Do NOT teach web scraping, RSS, automation, or "how to fetch news" unless bullets explicitly ask for it.
  Focus on summarizing events and implications.

Grounding policy:
- If mode == open_book:
  - Do NOT introduce any specific event/company/model/funding/policy claim unless it is supported by provided Evidence URLs.
  - For each event claim, attach a source as a Markdown link: ([Source](URL)).
  - Only use URLs provided in Evidence. If not supported, write: "Not found in provided sources."
- If requires_citations == true:
  - For outside-world claims, cite Evidence URLs the same way.
- Evergreen reasoning is OK without citations unless requires_citations is true.

Code:
- If requires_code == true, include at least one minimal, correct code snippet relevant to the bullets.

Style:
- Short paragraphs, bullets where helpful, code fences for code.
- Avoid fluff/marketing. Be precise and implementation-oriented.
"""

def worker_node(payload: dict) -> dict:
    task: Optional[Task] = None
    try:
        task = Task(**payload["task"])
        plan = Plan(**payload["plan"])
        evidence = [EvidenceItem(**e) for e in payload.get("evidence", [])]
        topic = payload["topic"]
        mode = payload.get("mode", "closed_book")
        logger.info("Worker started: task_id=%s title=%s", task.id, task.title)

        bullets_text = "\n- " + "\n- ".join(task.bullets)

        evidence_text = ""
        if evidence:
            evidence_text = "\n".join(
                f"- {e.title} | {e.url} | {e.published_at or 'date:unknown'}".strip()
                for e in evidence[:20]
            )

        section_md = llm.invoke(
            [
                SystemMessage(content=WORKER_SYSTEM),
                HumanMessage(
                    content=(
                        f"Blog title: {plan.blog_title}\n"
                        f"Audience: {plan.audience}\n"
                        f"Tone: {plan.tone}\n"
                        f"Blog kind: {plan.blog_kind}\n"
                        f"Constraints: {plan.constraints}\n"
                        f"Topic: {topic}\n"
                        f"Mode: {mode}\n\n"
                        f"Section title: {task.title}\n"
                        f"Goal: {task.goal}\n"
                        f"Target words: {task.target_words}\n"
                        f"Tags: {task.tags}\n"
                        f"requires_research: {task.requires_research}\n"
                        f"requires_citations: {task.requires_citations}\n"
                        f"requires_code: {task.requires_code}\n"
                        f"Bullets:{bullets_text}\n\n"
                        f"Evidence (ONLY use these URLs when citing):\n{evidence_text}\n"
                    )
                ),
            ]
        ).content.strip()

        logger.info("Worker completed: task_id=%s words=%d", task.id, len(section_md.split()))
        return {"sections": [(task.id, section_md)]}
    except Exception:
        if task is None:
            logger.exception("Worker node failed before task payload could be parsed")
        else:
            logger.exception("Worker node failed: task_id=%s title=%s", task.id, task.title)
        raise

In [ ]:
from pathlib import Path

def reducer_node(state: State) -> dict:
    try:
        plan = state["plan"]
        sections = state.get("sections", [])
        logger.info("Reducer started: title=%s sections=%d", plan.blog_title, len(sections))

        ordered_sections = [md for _, md in sorted(sections, key=lambda x: x[0])]
        body = "\n\n".join(ordered_sections).strip()
        final_md = f"# {plan.blog_title}\n\n{body}\n"

        filename = f"{plan.blog_title}.md"
        Path(filename).write_text(final_md, encoding="utf-8")

        logger.info("Reducer wrote output file: %s", filename)
        return {"final": final_md}
    except Exception:
        logger.exception("Reducer node failed")
        raise

In [ ]:
g = StateGraph(State)
g.add_node("router", router_node)
g.add_node("research", research_node)
g.add_node("orchestrator", orchestrator_node)
g.add_node("worker", worker_node)
g.add_node("reducer", reducer_node)

In [ ]:
g.add_edge(START, "router")
g.add_conditional_edges("router", route_next, {"research": "research", "orchestrator": "orchestrator"})
g.add_edge("research", "orchestrator")

g.add_conditional_edges("orchestrator", fanout, ["worker"])
g.add_edge("worker", "reducer")
g.add_edge("reducer", END)

In [ ]:
app = g.compile()
app

In [ ]:
def run(topic: str):
    try:
        logger.info("Invoking copilot graph: topic=%s", topic)
        out = app.invoke(
            {
                "topic": topic,
                "mode": "",
                "needs_research": False,
                "queries": [],
                "evidence": [],
                "plan": None,
                "sections": [],
                "final": "",
            }
        )

        logger.info("Copilot graph completed: topic=%s", topic)
        return out
    except ConfigurationError as exc:
        logger.error("Copilot graph blocked: %s", exc)
        raise
    except Exception:
        logger.exception("Copilot graph failed: topic=%s", topic)
        raise

In [ ]:
run("State of Multimodal LLMs in 2026")